In [1]:
import dotenv
dotenv.load_dotenv()

import os
from logfire.query_client import LogfireQueryClient

read_token = os.getenv('LOGFIRE_TOKEN_READONLY')
logfire_query_client = LogfireQueryClient(read_token=read_token)

In [2]:
trace_rows = logfire_query_client.query_json_rows(
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'agent run'
    ORDER BY start_timestamp DESC
    LIMIT 2
    """
)

In [3]:
trace_rows

{'columns': [{'name': 'trace_id', 'datatype': 'Utf8', 'nullable': False},
  {'name': 'start_timestamp',
   'datatype': {'Timestamp': ['Microsecond', 'UTC']},
   'nullable': False},
  {'name': 'duration', 'datatype': 'Float64', 'nullable': True}],
 'rows': [{'trace_id': '019e26cbd5b227a975eb2d22894520b1',
   'start_timestamp': '2026-05-14T14:04:33.812352Z',
   'duration': 3.816899477},
  {'trace_id': '019e26cbd5b227a975eb2d22894520b1',
   'start_timestamp': '2026-05-14T14:04:20.394475Z',
   'duration': 4.223590777}]}

In [4]:
trace_ids = [r['trace_id'] for r in trace_rows['rows']]
trace_ids

['019e26cbd5b227a975eb2d22894520b1', '019e26cbd5b227a975eb2d22894520b1']

In [5]:
trace_id = trace_ids[0]

run_row = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' as all_messages,
        attributes->>'gen_ai.usage.input_tokens' as input_tokens,
        attributes->>'gen_ai.usage.output_tokens' as output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'agent run'
    ORDER BY start_timestamp DESC
    LIMIT 1
    """
)

In [6]:
all_messages = run_row['rows'][0]
all_messages['all_messages'][0]

{'role': 'user',
 'parts': [{'type': 'text',
   'content': "Let's play 5 easy questions from Science & Nature"}]}

In [7]:
import trace_replay

In [12]:
from trace_replay.converter import fetch_trace, trace_to_run_result

In [11]:
trace = fetch_trace(trace_id, logfire_query_client)

In [14]:
run = trace_to_run_result(trace)

In [15]:
run.output

"The correct answer is **B) 244**.\n\nPlutonium (chemical symbol Pu) has several isotopes, but the most commonly referenced isotope for atomic weight is Plutonium-244. Atomic weight is a measure of the mass of an atom and is typically expressed in atomic mass units (amu). Plutonium-244 is significant in nuclear chemistry and is a long-lived isotope used in various applications, including nuclear reactors and weapons.\n\nYou've completed the trivia round! Let's calculate your final score. \n\nYou got:\n- Question 1: Correct\n- Question 2: Incorrect\n- Question 3: Correct\n- Question 4: Correct\n- Question 5: Incorrect\n\n### Your final score: 3 out of 5!\n\nGreat job! If you'd like to play again or explore different categories, just let me know!"

In [60]:
input_tokens = int(all_messages['input_tokens'])
output_tokens = int(all_messages['input_tokens'])

In [55]:
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

def calculate_cost(model_name, input_tokens, output_tokens):
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost

In [61]:
calculate_cost(model_name="gpt-4o-mini", input_tokens=input_tokens, output_tokens=output_tokens)

0.00096075